In [1]:
import pandas as pd
from datetime import datetime
import numpy as np
from sqlalchemy import create_engine

CONFIGURATION

In [2]:
CSV_FILE = "../data/sales_2.csv"

In [3]:
MYSQL_USER = "root"
MYSQL_PASSWORD = "curiousware@3loquent"
MYSQL_HOST = "localhost"
MYSQL_PORT = 3306
MYSQL_DATABASE = "etl_db"

TARGET_TABLE = "sales_etl_transformed"

Database Connection

In [4]:
import urllib.parse
conn_str = (
    f"mysql+pymysql://{MYSQL_USER}:{urllib.parse.quote(MYSQL_PASSWORD)}@{MYSQL_HOST}:{MYSQL_PORT}/{MYSQL_DATABASE}"
    )

In [5]:
engine = create_engine(conn_str)

STEP 1: READ CSV FILE (Extract)

In [6]:
try:
    df = pd.read_csv(CSV_FILE)
    print(f"Data extracted from {CSV_FILE}, shape: {df.shape}")
    display(df.head())
except Exception as e:
    print(f"Error reading CSV file: {e}")
    raise

Data extracted from ../data/sales_2.csv, shape: (10, 7)


,OrderId,Product,Category,SalesAmount,OrderDate,Region,CustomerName
0,1011,Monitor Stand,Furniture,55.00,2025-01-11,West,Kyle Reese
1,1012,Noise Cancelling Buds,Electronics,180.00,2025-01-11,East,Laura Croft
2,1013,Air Fryer Large,Home Goods,110.00,2025-01-12,Central,Michael Scott
3,1014,Ergonomic Mousepad,Electronics,15.99,2025-01-12,South,Nancy Drew
4,1015,4-Port USB Hub,Electronics,35.00,2025-01-13,East,Oliver Queen


STEP 2: TRANSFORMATIONS (Transform)

In [7]:
df.columns = (
    df.columns
      .str.replace(' ', '_')
      .str.replace(r'([A-Z])', r'_\1', regex=True)
      .str.lower()
      .str.strip('_')
)

In [8]:
# Convert order_date → datetime64
df["order_date"] = pd.to_datetime(df["order_date"])

In [9]:
df["sales_amount"] = pd.to_numeric(df["sales_amount"])

In [10]:
df["unit_price"] = df["sales_amount"]

In [11]:
conditions = [
    df["sales_amount"] >= 500,
    df["sales_amount"] >= 100
]
choice = ["High Value", "Medium Value"]

df["sales_tier"] = np.select(conditions, choice, default="Low Value")

In [12]:
df = df[df["sales_amount"] > 0]

In [13]:
df["load_timestamp"] = pd.to_datetime(datetime.utcnow())

C:\Users\chris\AppData\Local\Temp\ipykernel_9148\2612590778.py:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  df["load_timestamp"] = pd.to_datetime(datetime.utcnow())


In [14]:
print("Transformation complete.")
display(df.head())

Transformation complete.


,order_id,product,category,sales_amount,order_date,region,customer_name,unit_price,sales_tier,load_timestamp
0,1011,Monitor Stand,Furniture,55.00,2025-01-11,West,Kyle Reese,55.00,Low Value,2026-03-18 17:15:07.303463
1,1012,Noise Cancelling Buds,Electronics,180.00,2025-01-11,East,Laura Croft,180.00,Medium Value,2026-03-18 17:15:07.303463
2,1013,Air Fryer Large,Home Goods,110.00,2025-01-12,Central,Michael Scott,110.00,Medium Value,2026-03-18 17:15:07.303463
3,1014,Ergonomic Mousepad,Electronics,15.99,2025-01-12,South,Nancy Drew,15.99,Low Value,2026-03-18 17:15:07.303463
4,1015,4-Port USB Hub,Electronics,35.00,2025-01-13,East,Oliver Queen,35.00,Low Value,2026-03-18 17:15:07.303463


STEP 3: LOAD INTO MYSQL (Load)

In [15]:
if 'df' not in globals():
    raise NameError("df is not defined. Run the extract & transform cells first, or re-run the notebook from the top.")

try:
    df.to_sql(
        name=TARGET_TABLE,
        con=engine,
        if_exists='append',
        index=False,
        chunksize=1000
    )

    print(f"Data loaded into MySQL table '{TARGET_TABLE}' successfully.")

except Exception as e:
    print(f"Error loading data into MySQL: {e}")

Data loaded into MySQL table 'sales_etl_transformed' successfully.
